# Notebook 03 - Data Preprocessing

## Objective

Convert the raw FBref dataset into a clean dataset suitable for
feature engineering and machine learning.

Tasks
- Load configuration
- Remove goalkeepers
- Remove duplicate & derived statistics
- Select relevant features
- Merge transferred players
- Filter players by minutes
- Standardize positions
- Validate dataset
- Save cleaned dataset

Output:
data/processed/players_clean.csv

In [50]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [51]:
PROJECT_ROOT = Path("..")

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "players_data_light-2024_2025.csv"

FEATURE_CONFIG = PROJECT_ROOT / "config" / "feature_config.json"

PREPROCESSING_CONFIG = PROJECT_ROOT / "config" / "preprocessing_config.json"

df_raw = pd.read_csv(DATA_PATH)

df = df_raw.copy()

print(df.shape)

df.head()

(2854, 165)


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,G+A-PK,xG+xAG,PrgC,PrgP,PrgR,Sh,SoT,SoT%,Sh/90,SoT/90,G/Sh,G/SoT,Dist,FK,PK_stats_shooting,PKatt_stats_shooting,xG_stats_shooting,npxG_stats_shooting,npxG/Sh,G-xG,np:G-xG,Cmp,Att,Cmp%,TotDist,PrgDist,Ast_stats_passing,xAG_stats_passing,xA,A-xAG,KP,1/3,PPA,CrsPA,PrgP_stats_passing,Live,Dead,FK_stats_passing_types,TB,Sw,Crs,TI,CK,In,Out,Str,Cmp_stats_passing_types,Tkl,TklW,Def 3rd,Mid 3rd,Att 3rd,Att_stats_defense,Tkl%,Lost,Blocks_stats_defense,Sh_stats_defense,Pass,Int,Tkl+Int,Clr,Err,SCA,SCA90,PassLive,PassDead,TO,Sh_stats_gca,Fld,Def,GCA,GCA90,Touches,Def Pen,Def 3rd_stats_possession,Mid 3rd_stats_possession,Att 3rd_stats_possession,Att Pen,Live_stats_possession,Att_stats_possession,Succ,Succ%,Tkld,Tkld%,Carries,TotDist_stats_possession,PrgDist_stats_possession,PrgC_stats_possession,1/3_stats_possession,CPA,Mis,Dis,Rec,PrgR_stats_possession,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld_stats_misc,Off_stats_misc,Crs_stats_misc,Int_stats_misc,TklW_stats_misc,PKwon,PKcon,OG,Recov,Won,Lost_stats_misc,Won%,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,Cmp_stats_keeper_adv,Att_stats_keeper_adv,Cmp%_stats_keeper_adv,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,1,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,2000.0,3,1,86,1.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.00,1,8,3,0,0,NaN,0.00,0.0,NaN,NaN,NaN,0,0,0,0.0,0.0,NaN,0.0,0.0,50,63,79.4,887,361,0,0.0,0.0,0.0,0,8,0,0,8,51,12,2,0,0,2,10,0,0,0,0,50,2,2,1,1,0,1,100.0,0,3,1,2,1,3,0,0,2,2.09,2,0,0,0,0,0,0,0.0,73,2,19,40,15,0,73,2,0,0.0,1,50.0,41,152,68,1,0,0,1,0,40,3,0,0,0,0,2,0,2,1,2,0,0,0,7,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,2000.0,4,1,120,1.3,0,0,0,0,0,0,2,0,0.0,0.0,0.0,0.0,0.0,0.02,0,6,10,0,0,NaN,0.00,0.0,NaN,NaN,NaN,0,0,0,0.0,0.0,NaN,0.0,0.0,47,66,71.2,705,190,0,0.0,0.0,0.0,1,2,0,0,6,54,12,1,0,0,5,11,0,0,0,0,47,4,4,2,2,0,5,80.0,1,1,0,1,0,4,3,0,1,0.75,1,0,0,0,0,0,0,0.0,85,1,21,28,36,0,85,4,1,25.0,2,50.0,46,215,103,0,3,0,3,1,49,10,2,0,0,0,2,0,5,0,4,0,0,0,7,2,1,66.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,2003.0,1,1,65,0.7,0,0,0,0,0,0,1,0,0.1,0.1,0.0,0.1,0.0,0.10,3,2,3,1,0,0.0,1.38,0.0,0.0,NaN,24.5,0,0,0,0.1,0.1,0.07,-0.1,-0.1,17,29,58.6,268,110,0,0.0,0.0,0.0,0,0,0,0,2,21,8,0,0,0,1,8,0,0,0,0,17,3,2,2,1,0,3,100.0,0,1,0,1,1,4,0,0,0,0.00,0,0,0,0,0,0,0,0.0,36,1,8,19,9,1,36,1,1,100.0,0,0.0,13,101,67,3,2,1,0,2,16,3,1,0,0,2,0,1,1,1,2,0,0,0,2,0,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,James Abankwah,ie IRL,"DF,MF",Udinese,it Serie A,20.0,2004.0,6,0,88,1.0,0,0,0,0,0,0,1,0,0.1,0.1,0.0,0.1,0.0,0.06,3,4,1,1,0,0.0,1.02,0.0,0.0,NaN,15.0,0,0,0,0.1,0.1,0.06,-0.1,-0.1,36,46,78.3,614,206,0,0.0,0.0,0.0,0,2,0,0,4,45,1,1,0,0,0,0,0,0,0,0,36,4,2,4,0,0,3,66.7,1,2,1,1,1,5,3,0,1,1.02,1,0,0,0,0,0,0,0.0,65,8,37,22,7,2,65,0,0,NaN,0,NaN,29,219,165,3,1,1,1,3,34,1,1,0,0,4,3,0,0,1,2,0,0,0,7,2,2,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Keyliane Abdallah,fr FRA,FW,Marseille,fr Ligue 1,18.0,2006.0,1,0,3,0.0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.00,1,0,0,0,0,NaN,0.00,0.0,NaN,NaN,NaN,0,0,0,0.0,0.0,NaN,0.0,0.0,2,2,100.0,41,0,0,0.0,0.0,0.0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,2,1,1,1,0,0,1,100.0,0,0,0,0,0,1,0,0,0,0.00,0,0,0,0,0,0,0,0.0,4,0,3,1,0,0,4,0,0,NaN,0,NaN,1,10,9,1,0,0,1,0,3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print()

df.info()

Rows: 2854
Columns: 165

<class 'pandas.DataFrame'>
RangeIndex: 2854 entries, 0 to 2853
Columns: 165 entries, Rk to AvgDist
dtypes: float64(61), int64(99), str(5)
memory usage: 3.7 MB


In [53]:
with open("../config/feature_config.json") as f:
    feature_config = json.load(f)

with open("../config/preprocessing_config.json") as f:
    preprocessing_config = json.load(f)

In [54]:
ATTRIBUTE_MAP = feature_config["attribute_map"]

METADATA_COLUMNS = feature_config["metadata_columns"]

DISCIPLINE_COLUMNS = feature_config["discipline_columns"]

DUPLICATE_COLUMNS = feature_config["duplicate_columns"]

DERIVED_COLUMNS = feature_config["derived_columns"]

TEMPORARY_COLUMNS = feature_config["temporary_columns"]

In [55]:
def remove_goalkeepers(df):
    """
    Remove all goalkeeper records.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
    """

    print("Goalkeeper Removal")
    print("------------------")

    rows_before = len(df)

    df = df[
        ~df["Pos"].str.contains("GK", na=False)
    ].copy()

    rows_after = len(df)

    print(f"Rows before : {rows_before}")
    print(f"Rows after  : {rows_after}")
    print(f"Goalkeepers removed : {rows_before - rows_after}")

    # Verification
    assert not df["Pos"].str.contains("GK", na=False).any()

    return df

In [56]:
def drop_duplicate_columns(df, duplicate_columns):
    """
    Remove duplicate statistics columns.

    Parameters
    ----------
    df : pandas.DataFrame
    duplicate_columns : list

    Returns
    -------
    pandas.DataFrame
    """

    print("Duplicate Column Removal")
    print("------------------------")

    columns_before = len(df.columns)

    columns_to_drop = [
        col for col in duplicate_columns
        if col in df.columns
    ]

    missing_columns = [
        col for col in duplicate_columns
        if col not in df.columns
    ]

    df = df.drop(columns=columns_to_drop)

    columns_after = len(df.columns)

    print(f"Columns before : {columns_before}")
    print(f"Columns after  : {columns_after}")
    print(f"Columns removed : {columns_before - columns_after}")

    if missing_columns:
        print(f"Warning: Config columns not found: {missing_columns}")

    return df

In [57]:
def drop_derived_columns(df, derived_columns):
    """
    Remove derived statistics columns.

    Parameters
    ----------
    df : pandas.DataFrame
    derived_columns : list

    Returns
    -------
    pandas.DataFrame
    """

    print("Derived Column Removal")
    print("----------------------")

    columns_before = len(df.columns)

    columns_to_drop = [
        col for col in derived_columns
        if col in df.columns
    ]

    missing_columns = [
        col for col in derived_columns
        if col not in df.columns
    ]

    df = df.drop(columns=columns_to_drop)

    columns_after = len(df.columns)

    print(f"Columns before : {columns_before}")
    print(f"Columns after  : {columns_after}")
    print(f"Columns removed : {columns_before - columns_after}")

    if missing_columns:
        print(f"Warning: Config columns not found: {missing_columns}")

    return df

In [58]:

def select_columns(
    df,
    metadata_columns,
    discipline_columns,
    attribute_map,
):
    """
    Select metadata, discipline and relevant attribute features.
    """

    print("Feature Selection")
    print("-----------------")

    columns_before = len(df.columns)

    # Extract actual statistics from each attribute
    attribute_columns = []

    for attribute, details in attribute_map.items():
        attribute_columns.extend(details["features"])

    # Combine all required columns
    selected_columns = (
        metadata_columns
        + discipline_columns
        + attribute_columns
    )

    # Remove duplicates while preserving order
    selected_columns = list(dict.fromkeys(selected_columns))

    # Identify columns missing from the dataset
    missing_columns = [
        col for col in selected_columns
        if col not in df.columns
    ]

    # Keep only available columns
    available_columns = [
        col for col in selected_columns
        if col in df.columns
    ]

    df = df[available_columns].copy()

    print(f"Columns before: {columns_before}")
    print(f"Columns after: {len(df.columns)}")
    print(f"Selected columns: {len(available_columns)}")

    if missing_columns:
        print("Warning - Missing columns:", missing_columns)

    return df

In [59]:
# Check players with multiple records

player_records = (
    df.groupby("Player")
    .agg(
        Records=("Player", "size"),
        Clubs=("Squad", "nunique"),
        Total_Minutes=("Min", "sum"),
        Nations=("Nation", "nunique"),
        Birth_Years=("Born", "nunique")
    )
    .reset_index()
)

# Players appearing in multiple records
multiple_records = player_records[
    player_records["Records"] > 1
].sort_values("Records", ascending=False)

print("Unique players:", df["Player"].nunique())
print("Players with multiple records:", len(multiple_records))
print("Total records belonging to these players:",
      multiple_records["Records"].sum())

multiple_records.head(15)


Unique players: 2702
Players with multiple records: 152
Total records belonging to these players: 304


,Player,Records,Clubs,Total_Minutes,Nations,Birth_Years
19,Abdukodir Khusanov,2,2,1478,1,1
29,Adam Aznou,2,2,893,1,1
68,Akor Adams,2,2,1209,1,1
72,Albert Grønbaek,2,2,956,1,1
97,Alessandro Bianco,2,2,2618,1,1
103,Alessandro Marcandalli,2,2,400,1,1
104,Alessandro Vogliacco,2,2,1487,1,1
109,Alessio Zerbin,2,2,1520,1,1
129,Alexis Saelemaekers,2,2,1479,1,1
165,Amine Gouiri,2,2,2085,1,1


In [60]:
# Check whether repeated player names have consistent identity details

identity_check = (
    df.groupby("Player")
    .agg(
        Records=("Player", "size"),
        Clubs=("Squad", "nunique"),
        Nations=("Nation", "nunique"),
        Birth_Years=("Born", "nunique")
    )
    .reset_index()
)

identity_issues = identity_check[
    (identity_check["Records"] > 1) &
    (
        (identity_check["Nations"] > 1) |
        (identity_check["Birth_Years"] > 1)
    )
]

print("Repeated names with inconsistent identity details:",
      len(identity_issues))

identity_issues


Repeated names with inconsistent identity details: 5


,Player,Records,Clubs,Nations,Birth_Years
560,Danilo,2,2,1,2
584,David López,2,2,1,2
1273,Joshua King,2,2,2,2
1955,Nicolás González,2,2,2,2
2221,Rodri,2,2,1,2


In [61]:
# Players requiring identity verification

ambiguous_players = [
    "Danilo",
    "David López",
    "Joshua King",
    "Nicolás González",
    "Rodri"
]

identity_columns = [
    "Player",
    "Nation",
    "Born",
    "Age",
    "Squad",
    "Comp",
    "Pos",
    "Min"
]

ambiguous_records = (
    df[df["Player"].isin(ambiguous_players)]
    [identity_columns]
    .sort_values(["Player", "Born", "Squad"])
)

display(ambiguous_records)


,Player,Nation,Born,Age,Squad,Comp,Pos,Min
655,Danilo,br BRA,1991.0,33.0,Juventus,it Serie A,DF,581
654,Danilo,br BRA,2001.0,23.0,Nott'ham Forest,eng Premier League,"MF,FW",275
1516,David López,es ESP,1989.0,34.0,Girona,es La Liga,DF,2429
1515,David López,es ESP,2003.0,21.0,Mallorca,es La Liga,DF,169
1359,Joshua King,no NOR,1992.0,32.0,Toulouse,fr Ligue 1,"FW,MF",1673
1358,Joshua King,eng ENG,2007.0,17.0,Fulham,eng Premier League,MF,130
1036,Nicolás González,ar ARG,1998.0,26.0,Juventus,it Serie A,"FW,MF",1750
1035,Nicolás González,es ESP,2002.0,22.0,Manchester City,eng Premier League,MF,760
2226,Rodri,es ESP,1996.0,28.0,Manchester City,eng Premier League,MF,73
2227,Rodri,es ESP,2000.0,24.0,Betis,es La Liga,FW,118


In [62]:
# Check missing identity values

identity_columns = ["Player", "Nation", "Born"]

print("Missing identity values:")
print(df[identity_columns].isnull().sum())

# Check duplicate identity combinations
print(
    "\nDuplicate identity groups:",
    df.duplicated(subset=identity_columns).sum()
)

Missing identity values:
Player    0
Nation    7
Born      8
dtype: int64

Duplicate identity groups: 147


In [63]:
# Records with incomplete identity information

missing_identity = df[
    df[["Nation", "Born"]].isnull().any(axis=1)
]

display(
    missing_identity[
        ["Player", "Nation", "Born", "Age", "Squad", "Comp", "Pos", "Min"]
    ].sort_values("Player")
)

,Player,Nation,Born,Age,Squad,Comp,Pos,Min
1320,Atakan Karazor,NaN,1996.0,27.0,Stuttgart,de Bundesliga,MF,2618
1518,Fer López,NaN,NaN,NaN,Celta Vigo,es La Liga,"FW,MF",671
273,Hannes Behrens,de GER,NaN,NaN,Hoffenheim,de Bundesliga,DF,29
862,Jake Evans,NaN,NaN,NaN,Leicester City,eng Premier League,FW,24
1778,Jeremy Monga,NaN,NaN,NaN,Leicester City,eng Premier League,"FW,MF",112
1603,Mateus Mane,NaN,NaN,NaN,Wolves,eng Premier League,MF,2
1769,Max Moerstedt,de GER,NaN,NaN,Hoffenheim,de Bundesliga,FW,451
100,Olabade Aluko,NaN,NaN,NaN,Leicester City,eng Premier League,DF,2
663,Pape Daouda Diongue,sn SEN,NaN,NaN,Strasbourg,fr Ligue 1,"MF,FW",78
1917,Plamedi Nsingi,NaN,2000.0,23.0,Nantes,fr Ligue 1,MF,16


In [64]:
# Find repeated identity combinations
duplicate_identity = df[
    df.duplicated(
        subset=["Player", "Nation", "Born"],
        keep=False
    )
].sort_values(["Player", "Born", "Squad"])

display(
    duplicate_identity[
        ["Player", "Nation", "Born", "Squad", "Comp", "Pos", "Min"]
    ]
)

,Player,Nation,Born,Squad,Comp,Pos,Min
1352,Abdukodir Khusanov,uz UZB,2004.0,Lens,fr Ligue 1,DF,975
1351,Abdukodir Khusanov,uz UZB,2004.0,Manchester City,eng Premier League,DF,503
189,Adam Aznou,ma MAR,2006.0,Bayern Munich,de Bundesliga,DF,17
188,Adam Aznou,ma MAR,2006.0,Valladolid,es La Liga,DF,876
23,Akor Adams,ng NGA,2000.0,Montpellier,fr Ligue 1,FW,1075
...,...,...,...,...,...,...,...
536,Walid Cheddira,ma MAR,1998.0,Napoli,it Serie A,FW,12
359,Warren Bondo,fr FRA,2003.0,Milan,it Serie A,MF,162
358,Warren Bondo,fr FRA,2003.0,Monza,it Serie A,MF,1618
620,Woyo Coulibaly,ml MLI,1999.0,Leicester City,eng Premier League,DF,109


In [65]:
# Inspect columns required for aggregation

aggregation_columns = [
    "Player", "Nation", "Born", "Squad", "Comp", "Pos", "Min",
    "Gls", "Sh", "G/Sh",
    "Cmp", "Att", "Cmp%",
    "Won", "Lost", "Won%"
]

available = [
    col for col in aggregation_columns
    if col in df_raw.columns
]

missing = [
    col for col in aggregation_columns
    if col not in df_raw.columns
]

print("Available aggregation columns:")
print(available)

print("\nMissing aggregation columns:")
print(missing)

Available aggregation columns:
['Player', 'Nation', 'Born', 'Squad', 'Comp', 'Pos', 'Min', 'Gls', 'Sh', 'G/Sh', 'Cmp', 'Att', 'Cmp%', 'Won', 'Lost', 'Won%']

Missing aggregation columns:
[]


In [66]:
# Columns whose values should be summed across club stints

SUM_COLUMNS = [
    "Min",
    "Gls", "xG", "Sh", "SoT",
    "Ast", "xA", "KP", "SCA", "GCA", "TB",
    "Cmp", "Att", "PrgP", "PrgDist", "PPA", "Sw",
    "PrgC", "Carries", "CPA", "Succ",
    "Tkl", "TklW", "Int", "Blocks_stats_defense",
    "Clr", "Recov",
    "Mis", "Dis", "Rec",
    "Won", "Lost_stats_misc",
] + DISCIPLINE_COLUMNS

# Remove any repeated names from the list
SUM_COLUMNS = list(dict.fromkeys(SUM_COLUMNS))


def aggregate_player_records(df):
    """
    Aggregate multiple club records into one season-level
    record per verified player identity.

    Count statistics are summed, rate statistics are
    recalculated, and metadata is handled separately.
    """

    print("Transfer Aggregation")
    print("--------------------")

    rows_before = len(df)

    identity_columns = ["Player", "Nation", "Born"]

    required_columns = identity_columns + ["Min"]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    # Separate complete and incomplete identities
    complete_identity = df[
        df[identity_columns].notna().all(axis=1)
    ].copy()

    incomplete_identity = df[
        df[identity_columns].isna().any(axis=1)
    ].copy()

    sum_columns = [
        col for col in SUM_COLUMNS
        if col in df.columns
    ]

    def combine_metadata(group, main_record):

        # Preserve all unique clubs
        if "Squad" in group.columns:
            clubs = group["Squad"].dropna().unique()

            if len(clubs) > 0:
                main_record["Squad"] = ", ".join(sorted(clubs))

        # Preserve all unique competitions
        if "Comp" in group.columns:
            competitions = group["Comp"].dropna().unique()

            main_record["Comp"] = ", ".join(
                sorted(competitions)
            )
            if len(competitions) > 0:
                main_record["Comp"] = ", ".join(sorted(competitions))

        # Position associated with the most minutes
        if "Pos" in group.columns:
            position_minutes = (
                group.assign(
                    Min=pd.to_numeric(
                        group["Min"],
                        errors="coerce"
                    )
                )
                .groupby("Pos", dropna=True)["Min"]
                .sum()
            )

            if not position_minutes.empty:
                main_record["Pos"] = (
                    position_minutes.idxmax()
                )

        return main_record

    aggregated_records = []

    for _, group in complete_identity.groupby(
        identity_columns,
        sort=False
    ):

        group = group.copy()

        group["Min"] = pd.to_numeric(
            group["Min"],
            errors="coerce"
        )

        # Select record with highest minutes for base metadata
        valid_minutes = group["Min"].dropna()

        if valid_minutes.empty:
            main_record = group.iloc[0].copy()
        else:
            main_record = group.loc[
                valid_minutes.idxmax()
            ].copy()

        # Aggregate count statistics
        for col in sum_columns:
            main_record[col] = pd.to_numeric(
                group[col],
                errors="coerce"
            ).sum(min_count=1)

        # Handle descriptive metadata
        main_record = combine_metadata(
            group,
            main_record
        )

        # Recalculate completion percentage
        if {"Cmp", "Att"}.issubset(group.columns):
            attempts = main_record["Att"]

            main_record["Cmp%"] = (
                main_record["Cmp"] / attempts * 100
                if pd.notna(attempts) and attempts > 0
                else np.nan
            )

        # Recalculate duel win percentage
        if {"Won", "Lost_stats_misc"}.issubset(group.columns):
            total_duels = (
                main_record["Won"] +
                main_record["Lost_stats_misc"]
            )

            main_record["Won%"] = (
                main_record["Won"] / total_duels * 100
                if total_duels > 0
                else np.nan
            )

        # Recalculate goals per shot
        if {"Gls", "Sh"}.issubset(group.columns):
            shots = main_record["Sh"]

            main_record["G/Sh"] = (
                main_record["Gls"] / shots
                if pd.notna(shots) and shots > 0
                else np.nan
            )

        aggregated_records.append(main_record)

    aggregated_df = pd.DataFrame(aggregated_records)

    result = pd.concat(
        [aggregated_df, incomplete_identity],
        ignore_index=True
    )

    rows_after = len(result)

    print(f"Rows before aggregation: {rows_before}")
    print(f"Rows after aggregation: {rows_after}")
    print(f"Records combined: {rows_before - rows_after}")
    print(
        "Incomplete identity records preserved:",
        len(incomplete_identity)
    )

    print(
        "Unique players after aggregation:",
        result["Player"].nunique()
    )

    return result


In [67]:
# Step 1: Start with the raw dataset
df = df_raw.copy()

# Step 2: Remove goalkeepers
df = remove_goalkeepers(df)

# Step 3: Remove duplicate statistic columns
df = drop_duplicate_columns(
    df,
    DUPLICATE_COLUMNS
)

# Step 4: Remove derived columns
df = drop_derived_columns(
    df,
    DERIVED_COLUMNS
)

# Step 5: Aggregate transferred players
df = aggregate_player_records(df)

print("Shape after aggregation:", df.shape)

# Step 6: Select relevant features
df = select_columns(
    df,
    METADATA_COLUMNS,
    DISCIPLINE_COLUMNS,
    ATTRIBUTE_MAP
)

print("Shape after feature selection:", df.shape)


Goalkeeper Removal
------------------
Rows before : 2854
Rows after  : 2642
Goalkeepers removed : 212
Duplicate Column Removal
------------------------
Columns before : 165
Columns after  : 157
Columns removed : 8
Derived Column Removal
----------------------
Columns before : 157
Columns after  : 150
Columns removed : 7
Transfer Aggregation
--------------------
Rows before aggregation: 2642
Rows after aggregation: 2499
Records combined: 143
Incomplete identity records preserved: 10
Unique players after aggregation: 2494
Shape after aggregation: (2499, 150)
Feature Selection
-----------------
Columns before: 150
Columns after: 43
Selected columns: 43
Shape after feature selection: (2499, 43)


In [68]:
print("Shape:", df.shape)
print("Unique players:", df["Player"].nunique())

df[
    df["Player"].isin([
        "Abdukodir Khusanov",
        "Warren Bondo",
        "Woyo Coulibaly"
    ])
][["Player", "Squad", "Min", "Pos"]]

Shape: (2499, 43)
Unique players: 2494


,Player,Squad,Min,Pos
315,Warren Bondo,"Milan, Monza",1780,MF
545,Woyo Coulibaly,"Leicester City, Parma",1193,DF
1191,Abdukodir Khusanov,"Lens, Manchester City",1478,DF


In [69]:
MIN_MINUTES = 600

rows_before = len(df)

df = df[
    df["Min"] >= MIN_MINUTES
].copy()

print("Players before filtering:", rows_before)
print("Players after filtering:", len(df))
print("Players removed:", rows_before - len(df))

assert (df["Min"] >= MIN_MINUTES).all()

Players before filtering: 2499
Players after filtering: 1692
Players removed: 807


In [70]:
def standardize_positions(df):

    df = df.copy()

    df["Pos"] = (
        df["Pos"]
        .fillna("")
        .str.split(",")
        .str[0]
        .str.strip()
    )

    position_mapping = {
        "DF": "DF",
        "MF": "MF",
        "FW": "FW"
    }

    df["Pos"] = df["Pos"].map(position_mapping)

    return df


df = standardize_positions(df)

print(df["Pos"].value_counts(dropna=False))

Pos
DF    702
MF    564
FW    426
Name: count, dtype: int64


In [72]:
print("FINAL DATASET VALIDATION")
print("------------------------")

print("Dataset shape:", df.shape)

print(
    "Goalkeepers present:",
    df["Pos"].str.contains("GK", na=False).any()
)

print(
    "Players below 600 minutes:",
    (df["Min"] < 600).sum()
)

print(
    "Missing positions:",
    df["Pos"].isna().sum()
)

print(
    "Missing player names:",
    df["Player"].isna().sum()
)

print(
    "Missing minutes:",
    df["Min"].isna().sum()
)

print(
    "Duplicate rows:",
    df.duplicated().sum()
)

print(
    "Position categories:",
    sorted(df["Pos"].dropna().unique())
)

identity_columns = ["Player", "Nation", "Born"]

print(
    "Duplicate complete identities:",
    df.dropna(subset=identity_columns)
      .duplicated(subset=identity_columns)
      .sum()
)

FINAL DATASET VALIDATION
------------------------
Dataset shape: (1692, 43)
Goalkeepers present: False
Players below 600 minutes: 0
Missing positions: 0
Missing player names: 0
Missing minutes: 0
Duplicate rows: 0
Position categories: ['DF', 'FW', 'MF']
Duplicate complete identities: 0


In [73]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "players_clean.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(OUTPUT_PATH, index=False)

print("Dataset saved successfully!")
print("Location:", OUTPUT_PATH)

Dataset saved successfully!
Location: ..\data\processed\players_clean.csv
